In [50]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix

In [51]:
train_path = "../Datasets/kidney CT/Train"
test_path = "../Datasets/kidney CT/Test"
valid_path = "../Datasets/kidney CT/Valid"

In [52]:
train_dataaug = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range=12,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10
)
test_dataaug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)
vaild_dataaug = ImageDataGenerator(preprocessing_function=preprocess_input)

In [53]:
training_set = train_dataaug.flow_from_directory(
    train_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary"
)
validation_set = vaild_dataaug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary"
)
testing_set = test_dataaug.flow_from_directory(
    test_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary",
    shuffle = False
)

Found 8707 images belonging to 2 classes.
Found 1055 images belonging to 2 classes.
Found 1589 images belonging to 2 classes.


In [54]:
base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3),pooling="avg")

In [55]:
base_model.trainable = False

In [56]:
x = base_model.output
x = Dense(16, activation="relu", kernel_regularizer=l2(1e-5))(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

In [57]:
model = Model(inputs = base_model.input, outputs = output)

In [58]:
model.compile(optimizer=Adam(learning_rate=(1e-5)),loss="binary_crossentropy",metrics=['accuracy'])

In [59]:
early_stop = EarlyStopping(
    monitor="loss",
    patience=4,
    restore_best_weights=True
)

In [60]:
model.fit(training_set,validation_data=validation_set,epochs=7,callbacks=early_stop)

Epoch 1/7


I0000 00:00:1785386041.164635    1473 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_228908__.275


430/545 ━━━━━━━━━━━━━━━━━━━━ 15s 138ms/step - accuracy: 0.4728 - loss: 1.1011

I0000 00:00:1785386111.086259    1473 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_228908__.275


545/545 ━━━━━━━━━━━━━━━━━━━━ 126s 193ms/step - accuracy: 0.4956 - loss: 0.9648 - val_accuracy: 0.5649 - val_loss: 0.6956
Epoch 2/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 95s 174ms/step - accuracy: 0.5547 - loss: 0.7343 - val_accuracy: 0.6209 - val_loss: 0.6556
Epoch 3/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 87s 160ms/step - accuracy: 0.6069 - loss: 0.6537 - val_accuracy: 0.6095 - val_loss: 0.6477
Epoch 4/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 88s 161ms/step - accuracy: 0.6298 - loss: 0.6317 - val_accuracy: 0.6133 - val_loss: 0.6408
Epoch 5/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 84s 153ms/step - accuracy: 0.6571 - loss: 0.6080 - val_accuracy: 0.6303 - val_loss: 0.6241
Epoch 6/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.6705 - loss: 0.5979 - val_accuracy: 0.6417 - val_loss: 0.6083
Epoch 7/7
545/545 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.6873 - loss: 0.5831 - val_accuracy: 0.6739 - val_loss: 0.5890


In [61]:
prediction_prob = model.predict(testing_set)

100/100 ━━━━━━━━━━━━━━━━━━━━ 26s 166ms/step


In [62]:
result = (prediction_prob > 0.5).astype(int)
result 

array([[0],
       [0],
       [0],
       ...,
       [1],
       [1],
       [1]], shape=(1589, 1))

In [63]:
true_val = testing_set.classes
true_val

array([0, 0, 0, ..., 1, 1, 1], shape=(1589,), dtype=int32)

In [64]:
print(classification_report(true_val,result))
print(confusion_matrix(true_val,result))

              precision    recall  f1-score   support

           0       0.78      0.92      0.84       839
           1       0.88      0.71      0.79       750

    accuracy                           0.82      1589
   macro avg       0.83      0.81      0.81      1589
weighted avg       0.83      0.82      0.82      1589

[[769  70]
 [219 531]]


In [ ]:
model.save("kidney_CT_model1.keras")

In [ ]:
from tensorflow.keras.models import load_model
model = load_model("kidney_CT_model1.keras")

In [67]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [68]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [69]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

In [70]:
history = model.fit(training_set,validation_data=validation_set,epochs=10,callbacks=[ early_stop, reduce_lr])

Epoch 1/10


I0000 00:00:1785386756.750909    1476 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_292885__.307


493/545 ━━━━━━━━━━━━━━━━━━━━ 8s 170ms/step - accuracy: 0.6886 - loss: 0.5832

I0000 00:00:1785386858.342462    1474 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_292885__.307


545/545 ━━━━━━━━━━━━━━━━━━━━ 158s 241ms/step - accuracy: 0.7190 - loss: 0.5421 - val_accuracy: 0.7441 - val_loss: 0.4632 - learning_rate: 1.0000e-05
Epoch 2/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 83s 152ms/step - accuracy: 0.7748 - loss: 0.4552 - val_accuracy: 0.7697 - val_loss: 0.4216 - learning_rate: 1.0000e-05
Epoch 3/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 81s 149ms/step - accuracy: 0.8142 - loss: 0.4034 - val_accuracy: 0.7981 - val_loss: 0.3978 - learning_rate: 1.0000e-05
Epoch 4/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 85s 156ms/step - accuracy: 0.8307 - loss: 0.3635 - val_accuracy: 0.8161 - val_loss: 0.3744 - learning_rate: 1.0000e-05
Epoch 5/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 83s 151ms/step - accuracy: 0.8694 - loss: 0.3180 - val_accuracy: 0.8218 - val_loss: 0.3606 - learning_rate: 1.0000e-05
Epoch 6/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 85s 156ms/step - accuracy: 0.8822 - loss: 0.2911 - val_accuracy: 0.8645 - val_loss: 0.3064 - learning_rate: 1.0000e-05
Epoch 7/10
545/545 ━━━━━━━━━━━━━━━━━━━━ 87s 159ms/step -

In [71]:
prediction_prob = model.predict(testing_set)

100/100 ━━━━━━━━━━━━━━━━━━━━ 28s 154ms/step


In [72]:
result1 = (prediction_prob > 0.5).astype(int)
result1 

array([[0],
       [0],
       [0],
       ...,
       [1],
       [1],
       [1]], shape=(1589, 1))

In [73]:
print(classification_report(true_val,result1))
print(confusion_matrix(true_val,result1))

              precision    recall  f1-score   support

           0       0.93      0.94      0.93       839
           1       0.93      0.92      0.93       750

    accuracy                           0.93      1589
   macro avg       0.93      0.93      0.93      1589
weighted avg       0.93      0.93      0.93      1589

[[785  54]
 [ 58 692]]


In [ ]:
model.save("kidney_CT_model.keras")